# Governed Agentic RAG — run on Google Colab (T4 GPU)

Run the cells **top to bottom**. Assumes you've already `git clone`d the repo into Colab.

**Before you start:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*.

What this does: install deps → install + start Ollama → pull `gemma4:12b` → build the
index → run the governance ablation (with RAGAS faithfulness, meaningful on the 12B judge)
→ show the metrics and plots.

## 1. Confirm the GPU (should show a Tesla T4)

In [1]:
!nvidia-smi

Mon Aug  3 14:19:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Enter the repo
Change `REPO_DIR` if you cloned to a different path.

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
import os
REPO_DIR = "/content/drive/MyDrive/governed-agentic-rag"   # <-- change if you cloned elsewhere
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
assert os.path.exists("scripts/build_index.py"), "Not in the repo root — fix REPO_DIR"

cwd: /content/drive/MyDrive/governed-agentic-rag


In [13]:
pwd

'/content/drive/MyDrive/governed-agentic-rag'

## 3. Install Python dependencies (Colab-specific)

Uses `requirements-colab.txt`, which **omits torch/numpy/pandas/matplotlib** (Colab already
has them — reinstalling torch can break the GPU) and **omits langgraph** (unused by the eval).
First we remove Colab's preinstalled `langgraph 1.x`, which otherwise conflicts with the
`langchain-core 0.3.x` that ragas needs.

Harmless leftover warnings about `requests`/`cryptography` versions can be ignored.

In [14]:
# remove Colab's preinstalled langgraph 1.x (unused here, conflicts with langchain-core 0.3.x)
!pip uninstall -y -q langgraph langgraph-prebuilt langgraph-checkpoint langgraph-sdk
# install the Colab-tuned deps
!pip install -q -r requirements-colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 90.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## 4. Install Ollama, start the server, pull the model
`ollama serve` runs in the background; we wait a few seconds, then pull `gemma4:12b`
(~8 GB, fits the T4). This can take a few minutes the first time.

In [16]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull gemma4:12b
!ollama list

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (304 kB/s)                     
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and di

## 5. Choose where files live + which model
**Recommended:** mount Google Drive so the index + results persist across sessions, and
select the `colab` profile (which sets Drive paths **and** `gemma4:12b`).

Env vars set here are inherited by the `!python ...` cells below.

In [17]:

import os
os.environ["GRAG_PROFILE"] = "colab"   # paths -> Drive, model -> gemma4:12b

# --- ephemeral alternative (no Drive; lost when the VM recycles) ---
# os.environ["GRAG_BASE_DIR"] = "/content/governed_rag"
# os.environ["GRAG_MODEL"]    = "gemma4:12b"

import sys; sys.path.insert(0, ".")
from src.config import load_config
cfg = load_config()
print("profile:", cfg["_profile"], "| model:", cfg["llm"]["model"])
print("qdrant :", cfg["paths"]["qdrant_dir"])
print("faithfulness backend:", cfg["evaluation"]["faithfulness_backend"])

profile: colab | model: gemma4:12b
qdrant : /content/drive/MyDrive/governed_rag/qdrant
faithfulness backend: ragas


## 6. Build the index (once)
On the T4 this is fast (GPU embedding). If you mounted Drive and already built it in a
previous session, `index_status.py` will show the count and you can **skip the build**.

In [18]:
!python scripts/index_status.py
# If it says 'No collection ...', build it:
!python scripts/build_index.py

Collection 'governed_rag' @ /content/drive/MyDrive/governed_rag/qdrant
  total chunks indexed: 5150
  poisoned chunks     : 4
  by classification   : {'restricted': 1540, 'internal': 2578, 'public': 1032}
  by department       : {'legal': 1307, 'engineering': 1231, 'finance': 1323, 'hr': 1289}
14:58:38 | INFO    | Logging to /content/drive/MyDrive/governed_rag/artifacts/logs/build_index.log
14:58:38 | INFO    | Building index (downloads HotpotQA + embedding model on first run)...
14:59:05 | INFO    | Loading HotpotQA subset (500 questions)...
README.md: 100% 9.52k/9.52k [00:00<00:00, 18.3MB/s]
README.md: 100% 9.52k/9.52k [00:00<00:00, 25.6MB/s]

distractor/train-00000-of-00002.parquet: downloading bytes:   3% 4.33M/166M [00:01<01:06, 2.43MB/s]
distractor/train-00000-of-00002.parquet: downloading bytes:  10% 16.5M/166M [00:01<00:13, 11.3MB/s,  417kB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes:  27% 44.5M/166M [00:02<00:03, 33.8MB/s, 2.23MB/s  ] ]
distractor/train-000

## 7. Run the governance ablation
6 configs × (boundary=5, poison=4, qa=8) with RAGAS faithfulness on the 12B judge.
Much faster than CPU. Bump `--n-boundary` to 10–20 for smoother numbers if you like.

In [ ]:
!python scripts/run_eval.py --n-boundary 5

16:21:17 | INFO    | Logging to /content/drive/MyDrive/governed_rag/artifacts/logs/run_eval.log
Loading weights: 100% 199/199 [00:00<00:00, 989.80it/s]
16:21:43 | INFO    | Building test suite from the indexed corpus...
16:21:44 | INFO    |   boundary=5 poison=4 pii_probes=20
16:21:44 | INFO    | Ablation: 6 configs x (boundary=5, poison=4) ragas=True
16:21:44 | INFO    | === config: baseline ===
16:28:34 | INFO    |   leak=1.00 inj=0.00 poison_exp=1.00 audit=0.00 faith=0.00 lat=21726ms tok=1025
16:28:34 | INFO    | === config: +C1 permission ===


## 8. View the results (metrics + plots)

In [ ]:
import json, os
from IPython.display import Image, display
from src.config import load_config
art = load_config()["paths"]["artifacts_dir"]
print(json.dumps(json.load(open(os.path.join(art, "metrics.json"))), indent=2))
display(Image(os.path.join(art, "tradeoff.png")))
display(Image(os.path.join(art, "safety_bars.png")))

## 9. (Optional) demo — same query, governance on vs off
Run the walkthrough notebook cells, or open `notebooks/demo.ipynb`.

**For your report:** copy the values from `metrics.json` into the tables in
`reports/report.md`, and cite the model as `gemma4:12b` (the one these numbers came from).